# Week 2 — one raw tool call

This notebook is where the four modules you write this week get put together. It
holds no logic of its own: every line calls something you wrote in
`src/fin_agent/`, which is the point. Logic in a notebook cannot be tested, and
`pytest tests/week_02` is what says whether you are done.

The assignment is [`docs/weeks/week-02/spec.md`](../docs/weeks/week-02/spec.md).
Read it first. This is the last step, not the first one.

## Write these four first

In this order, running the matching tests after each — one change, one run:

| File | Tests | Count |
| --- | --- | --- |
| `src/fin_agent/tools/quote.py` | `pytest tests/week_02/test_quote.py` | 21 |
| `src/fin_agent/ledger.py` | `pytest tests/week_02/test_ledger.py` | 8 |
| `src/fin_agent/llm.py` then `dispatch.py` | `pytest tests/week_02/test_dispatch.py` | 19 |

Until those exist, the next cell raises `ModuleNotFoundError` naming the one you
have not written. That is the correct first result.

In [ ]:
from fin_agent import config
from fin_agent.dispatch import parse_tool_call, run_tool_call
from fin_agent.ledger import log_run
from fin_agent.llm import call_model
from fin_agent.tools.quote import get_latest_close

config.MODEL

## The tool schema

Write this by hand. It is the only description of your tool the model ever sees,
and in week 3 you will watch a vague description get a tool misused — which lands
harder if you wrote a good one first.

One dict, in the provider's function format: a name, a one-line description, and
a `ticker` parameter. The spec says what goes in it; this cell stays empty until
you fill it.

In [ ]:
TOOLS = [
    # TODO: one function schema for get_latest_close
]

assert TOOLS, "write the schema before going on"

## The question

Two messages: a system message telling the model what it is and which tickers
exist, and the user's question. Keep the system message short — you will move it
out to markdown in week 6 and a long one is harder to externalise.

In [ ]:
MESSAGES = [
    # TODO: a system message, then the user's question
]

assert MESSAGES, "write the messages before going on"

## The call

One `requests.post`. Nothing is parsed here — `call_model` returns the provider's
JSON unchanged, and you should look at it before anything consumes it. The shape
of `choices[0].message.tool_calls` is what the next cell navigates.

**No model key this week?** Run the cell below it instead. The parsing, dispatch
and ledger code is identical; only `call_model` is skipped. Say in your pull
request which one you ran.

In [ ]:
response = call_model(MESSAGES, TOOLS)
response

In [ ]:
# Without a key: the same thing, recorded.
import json
from pathlib import Path

recorded = Path("../tests/fixtures/model/tool_call.json")
response = json.loads(recorded.read_text(encoding="utf-8"))
response

## The decision

The week in one cell. The model asked for something; `parse_tool_call` decides
whether that request is one your code is willing to recognise, and raises rather
than guessing if it is not.

`allowed` is what the model may ask for. `registry` is what this process will
actually execute. They look redundant today and become the pre-tool hook in
week 4.

In [ ]:
ALLOWED = {"get_latest_close"}
REGISTRY = {"get_latest_close": get_latest_close}

call = parse_tool_call(response, ALLOWED)
call

In [ ]:
result = run_tool_call(call, REGISTRY) if call else None
result

## Make it refuse

Do this now rather than trusting the tests. Three recorded responses in
`tests/fixtures/model/` are things a model can legitimately emit after reading a
page someone else wrote. Each should raise.

Watch `unknown_tool.json` in particular: it asks for `run_shell_command` with
`cat .env`. Your `allowed` set is the only thing standing between that request
and a process willing to honour it.

In [ ]:
from pathlib import Path

for name in ("unknown_tool", "bad_arguments", "non_object_arguments"):
    payload = json.loads(Path(f"../tests/fixtures/model/{name}.json").read_text(encoding="utf-8"))
    try:
        parse_tool_call(payload, ALLOWED)
    except ValueError as exc:
        print(f"{name}: refused — {exc}")
    else:
        print(f"{name}: NOT REFUSED — this is the security criterion failing")

## The cost

`prompt_tokens` and `completion_tokens` come out of `response["usage"]`. Work
`cost_usd` out by hand from the provider's published price — doing that
arithmetic once is worth more than a lookup table, and by week 5 you will be
reading this file to tell whether compaction worked.

`ledger.csv` is gitignored, so the file stays yours. The row goes in the cost
ledger table in your pull request.

In [ ]:
usage = response["usage"]

log_run(
    week="02",
    model=response["model"],
    calls=1,
    tokens_in=usage["prompt_tokens"],
    tokens_out=usage["completion_tokens"],
    cost_usd=0.0,  # TODO: work this out from the published price
    note="first raw tool call",
)

## Before you open the pull request

- `pytest tests/week_02` — 48 passing
- The security criterion: the three refusal cells above printed "refused"
- Your cost ledger row, in the pull request table
- Your prompt trail, including the prompts that went wrong — this is the item
  that carries the most weight
- Sprint 1's consultant section, since the boundary falls this week

Outputs are stripped on commit by `nbstripout`, so what you push is the code you
wrote rather than the numbers it produced. Put the numbers in the pull request.